In [1]:
import requests
import pandas as pd
from datetime import datetime
import calendar
import tomllib
import time

# Database and Spark Imports
from cassandra.cluster import Cluster
from pymongo import MongoClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp, col
import time
import toml
from pyspark.sql import SparkSession
from cassandra.cluster import Cluster
from pymongo.server_api import ServerApi


In [2]:
secrets = toml.load(".streamlit/secrets.toml")
uri = secrets["database"]["uri"]
client = MongoClient(uri, server_api=ServerApi('1'))  

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [3]:
def fetch_elhub_data(dataset, years, base_url="https://api.elhub.no/energy-data/v0/price-areas"):
    """
    Fetches Elhub data for a given dataset and list of years.
    Returns a Pandas DataFrame with all results.
    """
    all_rows = []
    failures = []

    # Determine the correct JSON key for the dataset
    # E.g., "PRODUCTION_PER_GROUP_MBA_HOUR" -> "productionPerGroupMbaHour"
    def dataset_json_key(ds):
        parts = ds.lower().split('_')
        camel = parts[0] + ''.join(word.capitalize() for word in parts[1:])
        return camel

    list_key = dataset_json_key(dataset)

    for year in years:
        for m in range(1, 13):
            start = datetime(year, m, 1, 0, 0, 0)
            end_day = calendar.monthrange(year, m)[1]
            end = datetime(year, m, end_day, 23, 59, 59)
            start_str = start.strftime("%Y-%m-%dT%H:%M:%S") + "%2B01:00"
            end_str   = end.strftime("%Y-%m-%dT%H:%M:%S") + "%2B01:00"
            url = f"{base_url}?dataset={dataset}&startDate={start_str}&endDate={end_str}"

            r = requests.get(url, timeout=60)
            if r.ok:
                j = r.json()
                month_rows = [
                    rec
                    for item in j.get("data", [])
                    for rec in item.get("attributes", {}).get(list_key, [])
                ]
                all_rows.extend(month_rows)
                print(f"{year}-{m:02d}: {len(month_rows)} rows added")
            else:
                failures.append((year, m, r.status_code))
                print(f"{year}-{m:02d}: HTTP {r.status_code}")

    print(f"\nTotal rows: {len(all_rows)} | Failed months: {len(failures)}")
    df = pd.DataFrame(all_rows)
    if not df.empty:
        df.columns = [c.lower() for c in df.columns]
        display(df.head())
    else:
        print("No data returned.")
    return df

In [4]:
years = [2021, 2022, 2023, 2024]
dataset = "PRODUCTION_PER_GROUP_MBA_HOUR"
df_production = fetch_elhub_data(dataset, years)

dataset = "CONSUMPTION_PER_GROUP_MBA_HOUR"
df_consumption = fetch_elhub_data(dataset, years)

2021-01: 17856 rows added
2021-02: 16128 rows added
2021-03: 17832 rows added
2021-04: 17280 rows added
2021-05: 17856 rows added
2021-06: 17976 rows added
2021-07: 18600 rows added
2021-08: 18600 rows added
2021-09: 18000 rows added
2021-10: 18625 rows added
2021-11: 18000 rows added
2021-12: 18600 rows added
2022-01: 18600 rows added
2022-02: 16800 rows added
2022-03: 18575 rows added
2022-04: 18000 rows added
2022-05: 18600 rows added
2022-06: 18000 rows added
2022-07: 18600 rows added
2022-08: 18600 rows added
2022-09: 18000 rows added
2022-10: 18625 rows added
2022-11: 18000 rows added
2022-12: 18600 rows added
2023-01: 18600 rows added
2023-02: 16800 rows added
2023-03: 18575 rows added
2023-04: 18000 rows added
2023-05: 18600 rows added
2023-06: 18000 rows added
2023-07: 18600 rows added
2023-08: 18600 rows added
2023-09: 18000 rows added
2023-10: 18625 rows added
2023-11: 18000 rows added
2023-12: 18600 rows added
2024-01: 18600 rows added
2024-02: 17400 rows added
2024-03: 185

,endtime,lastupdatedtime,pricearea,productiongroup,quantitykwh,starttime
0,2021-01-01T01:00:00+01:00,2024-12-20T10:35:40+01:00,NO1,hydro,2507716.8,2021-01-01T00:00:00+01:00
1,2021-01-01T02:00:00+01:00,2024-12-20T10:35:40+01:00,NO1,hydro,2494728.0,2021-01-01T01:00:00+01:00
2,2021-01-01T03:00:00+01:00,2024-12-20T10:35:40+01:00,NO1,hydro,2486777.5,2021-01-01T02:00:00+01:00
3,2021-01-01T04:00:00+01:00,2024-12-20T10:35:40+01:00,NO1,hydro,2461176.0,2021-01-01T03:00:00+01:00
4,2021-01-01T05:00:00+01:00,2024-12-20T10:35:40+01:00,NO1,hydro,2466969.2,2021-01-01T04:00:00+01:00


2021-01: 18600 rows added
2021-02: 16800 rows added
2021-03: 18575 rows added
2021-04: 18000 rows added
2021-05: 18600 rows added
2021-06: 18000 rows added
2021-07: 18600 rows added
2021-08: 18600 rows added
2021-09: 18000 rows added
2021-10: 18625 rows added
2021-11: 18000 rows added
2021-12: 18600 rows added
2022-01: 18600 rows added
2022-02: 16800 rows added
2022-03: 18575 rows added
2022-04: 18000 rows added
2022-05: 18600 rows added
2022-06: 18000 rows added
2022-07: 18600 rows added
2022-08: 18600 rows added
2022-09: 18000 rows added
2022-10: 18625 rows added
2022-11: 18000 rows added
2022-12: 18600 rows added
2023-01: 18600 rows added
2023-02: 16800 rows added
2023-03: 18575 rows added
2023-04: 18000 rows added
2023-05: 18600 rows added
2023-06: 18000 rows added
2023-07: 18600 rows added
2023-08: 18600 rows added
2023-09: 18000 rows added
2023-10: 18625 rows added
2023-11: 18000 rows added
2023-12: 18600 rows added
2024-01: 18600 rows added
2024-02: 17400 rows added
2024-03: 185

,consumptiongroup,endtime,lastupdatedtime,meteringpointcount,pricearea,quantitykwh,starttime
0,cabin,2021-01-01T01:00:00+01:00,2024-12-20T10:35:40+01:00,100607,NO1,177071.56,2021-01-01T00:00:00+01:00
1,cabin,2021-01-01T02:00:00+01:00,2024-12-20T10:35:40+01:00,100607,NO1,171335.12,2021-01-01T01:00:00+01:00
2,cabin,2021-01-01T03:00:00+01:00,2024-12-20T10:35:40+01:00,100607,NO1,164912.02,2021-01-01T02:00:00+01:00
3,cabin,2021-01-01T04:00:00+01:00,2024-12-20T10:35:40+01:00,100607,NO1,160265.77,2021-01-01T03:00:00+01:00
4,cabin,2021-01-01T05:00:00+01:00,2024-12-20T10:35:40+01:00,100607,NO1,159828.69,2021-01-01T04:00:00+01:00


In [5]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = "python"
print(f"JAVA_HOME set to: {os.environ['JAVA_HOME']}")

# Verify Java is accessible
import subprocess
try:
    result = subprocess.run([f"{os.environ['JAVA_HOME']}/bin/java", "-version"], 
                          capture_output=True, text=True, timeout=5)
    print(f"Java version: {result.stderr.split(chr(10))[0]}")
except Exception as e:
    print(f"Error checking Java: {e}")

JAVA_HOME set to: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
Java version: openjdk version "17.0.17" 2025-10-21


In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('SparkCassandraApp').\
    config('spark.jars.packages', 'com.datastax.spark:spark-cassandra-connector_2.12:3.5.1,org.mongodb.spark:mongo-spark-connector_2.12:10.2.0').\
    config('spark.cassandra.connection.host', 'localhost').\
    config('spark.sql.extensions', 'com.datastax.spark.connector.CassandraSparkExtensions').\
    config('spark.sql.catalog.mycatalog', 'com.datastax.spark.connector.datasource.CassandraCatalog').\
    config('spark.cassandra.connection.port', '9042').getOrCreate()
    
print("Spark session connected to Cassandra.")

:: loading settings :: url = jar:file:/Users/henrikengdal/Documents/GitHub/IND320-HenrikEngdal-Project/.conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/henrikengdal/.ivy2/cache
The jars for the packages stored in: /Users/henrikengdal/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c68a84d4-da1c-47c6-9cd8-78ca720f3119;1.0
	confs: [default]
	found com.datastax.spark#spark-cassandra-connector_2.12;3.5.1 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.1 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found org.apache.cassandra#java-driver-core-shaded;4.18.1 in central
	found com.datastax.oss#native-protocol;1.5.1 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogr

Spark session connected to Cassandra.


In [7]:
# Convert Pandas DataFrame to Spark DataFrame
df_spark_production = spark.createDataFrame(df_production)

# Show the schema and a few rows
df_spark_production.printSchema()
df_spark_production.show(5)

root
 |-- endtime: string (nullable = true)
 |-- lastupdatedtime: string (nullable = true)
 |-- pricearea: string (nullable = true)
 |-- productiongroup: string (nullable = true)
 |-- quantitykwh: double (nullable = true)
 |-- starttime: string (nullable = true)



25/11/26 22:37:26 WARN TaskSetManager: Stage 0 contains a task of very large size (11700 KiB). The maximum recommended task size is 1000 KiB.
25/11/26 22:37:31 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 0 (TID 0): Attempting to kill Python Worker


+--------------------+--------------------+---------+---------------+-----------+--------------------+
|             endtime|     lastupdatedtime|pricearea|productiongroup|quantitykwh|           starttime|
+--------------------+--------------------+---------+---------------+-----------+--------------------+
|2021-01-01T01:00:...|2024-12-20T10:35:...|      NO1|          hydro|  2507716.8|2021-01-01T00:00:...|
|2021-01-01T02:00:...|2024-12-20T10:35:...|      NO1|          hydro|  2494728.0|2021-01-01T01:00:...|
|2021-01-01T03:00:...|2024-12-20T10:35:...|      NO1|          hydro|  2486777.5|2021-01-01T02:00:...|
|2021-01-01T04:00:...|2024-12-20T10:35:...|      NO1|          hydro|  2461176.0|2021-01-01T03:00:...|
|2021-01-01T05:00:...|2024-12-20T10:35:...|      NO1|          hydro|  2466969.2|2021-01-01T04:00:...|
+--------------------+--------------------+---------+---------------+-----------+--------------------+
only showing top 5 rows



In [8]:
# Convert Pandas DataFrame to Spark DataFrame
df_spark_consumption = spark.createDataFrame(df_consumption)

# Show the schema and a few rows
df_spark_consumption.printSchema()
df_spark_consumption.show(5)

root
 |-- consumptiongroup: string (nullable = true)
 |-- endtime: string (nullable = true)
 |-- lastupdatedtime: string (nullable = true)
 |-- meteringpointcount: long (nullable = true)
 |-- pricearea: string (nullable = true)
 |-- quantitykwh: double (nullable = true)
 |-- starttime: string (nullable = true)



25/11/26 22:37:42 WARN TaskSetManager: Stage 1 contains a task of very large size (12461 KiB). The maximum recommended task size is 1000 KiB.


+----------------+--------------------+--------------------+------------------+---------+-----------+--------------------+
|consumptiongroup|             endtime|     lastupdatedtime|meteringpointcount|pricearea|quantitykwh|           starttime|
+----------------+--------------------+--------------------+------------------+---------+-----------+--------------------+
|           cabin|2021-01-01T01:00:...|2024-12-20T10:35:...|            100607|      NO1|  177071.56|2021-01-01T00:00:...|
|           cabin|2021-01-01T02:00:...|2024-12-20T10:35:...|            100607|      NO1|  171335.12|2021-01-01T01:00:...|
|           cabin|2021-01-01T03:00:...|2024-12-20T10:35:...|            100607|      NO1|  164912.02|2021-01-01T02:00:...|
|           cabin|2021-01-01T04:00:...|2024-12-20T10:35:...|            100607|      NO1|  160265.77|2021-01-01T03:00:...|
|           cabin|2021-01-01T05:00:...|2024-12-20T10:35:...|            100607|      NO1|  159828.69|2021-01-01T04:00:...|
+---------------

25/11/26 22:37:46 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 1 (TID 1): Attempting to kill Python Worker


In [10]:
from pyspark.sql.functions import col

df_spark_production = (
    df_spark_production
    .withColumn("starttime", col("starttime").cast("timestamp"))
    .withColumn("endtime", col("endtime").cast("timestamp"))
    .withColumn("lastupdatedtime", col("lastupdatedtime").cast("timestamp"))
)

df_spark_production.printSchema()

root
 |-- endtime: timestamp (nullable = true)
 |-- lastupdatedtime: timestamp (nullable = true)
 |-- pricearea: string (nullable = true)
 |-- productiongroup: string (nullable = true)
 |-- quantitykwh: double (nullable = true)
 |-- starttime: timestamp (nullable = true)



In [11]:
df_spark_consumption = (
    df_spark_consumption
    .withColumn("starttime", col("starttime").cast("timestamp"))
    .withColumn("endtime", col("endtime").cast("timestamp"))
    .withColumn("lastupdatedtime", col("lastupdatedtime").cast("timestamp"))
)

df_spark_consumption.printSchema()

root
 |-- consumptiongroup: string (nullable = true)
 |-- endtime: timestamp (nullable = true)
 |-- lastupdatedtime: timestamp (nullable = true)
 |-- meteringpointcount: long (nullable = true)
 |-- pricearea: string (nullable = true)
 |-- quantitykwh: double (nullable = true)
 |-- starttime: timestamp (nullable = true)



In [12]:
keyspace_name = "ind320_project"
cluster = Cluster(['localhost'], port=9042)
session = cluster.connect()

# Ensure the keyspace exists (idempotent operation)
session.execute(f"""
CREATE KEYSPACE IF NOT EXISTS {keyspace_name}
WITH replication = {{'class':'SimpleStrategy', 'replication_factor' : 1}};
""")

session.set_keyspace(keyspace_name)
print(f"Cassandra keyspace: {keyspace_name} created successfully!")

# Create the production table
session.execute("""
CREATE TABLE IF NOT EXISTS production (
    pricearea text,
    productiongroup text,
    starttime timestamp,
    endtime timestamp,
    quantitykwh double,
    lastupdatedtime timestamp,
    PRIMARY KEY (pricearea, starttime, productiongroup)
);
""")
print("Cassandra table production created successfully!")

# Create the consumption table
session.execute("""
CREATE TABLE IF NOT EXISTS consumption (
    pricearea text,
    consumptiongroup text,
    starttime timestamp,
    endtime timestamp,
    quantitykwh double,
    lastupdatedtime timestamp,
    meteringpointcount int,
    PRIMARY KEY (pricearea, starttime, consumptiongroup)
);
""")

print("Cassandra table consumption created successfully!")

Cassandra keyspace: ind320_project created successfully!
Cassandra table production created successfully!
Cassandra table consumption created successfully!


In [ ]:

# Insert the Spark DataFrame into Cassandra with error handling and schema debug
try:
    print("Spark DataFrame schema before write:")
    df_spark_production.printSchema()
    print("First 5 rows:")
    df_spark_production.show(5)

    (
        df_spark_production.write
        .format("org.apache.spark.sql.cassandra")
        .mode("append")
        .options(table="production", keyspace=keyspace_name)
        .save()
    )
    print("Production data successfully inserted into Cassandra!")
except Exception as e:
    import traceback
    print("Error while writing to Cassandra:")
    traceback.print_exc()
    print(f"Exception: {e}")

Spark DataFrame schema before write:
root
 |-- endtime: timestamp (nullable = true)
 |-- lastupdatedtime: timestamp (nullable = true)
 |-- pricearea: string (nullable = true)
 |-- productiongroup: string (nullable = true)
 |-- quantitykwh: double (nullable = true)
 |-- starttime: timestamp (nullable = true)

First 5 rows:


25/11/26 22:39:24 WARN TaskSetManager: Stage 2 contains a task of very large size (11700 KiB). The maximum recommended task size is 1000 KiB.
25/11/26 22:39:28 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 2 (TID 2): Attempting to kill Python Worker


+-------------------+-------------------+---------+---------------+-----------+-------------------+
|            endtime|    lastupdatedtime|pricearea|productiongroup|quantitykwh|          starttime|
+-------------------+-------------------+---------+---------------+-----------+-------------------+
|2021-01-01 01:00:00|2024-12-20 10:35:40|      NO1|          hydro|  2507716.8|2021-01-01 00:00:00|
|2021-01-01 02:00:00|2024-12-20 10:35:40|      NO1|          hydro|  2494728.0|2021-01-01 01:00:00|
|2021-01-01 03:00:00|2024-12-20 10:35:40|      NO1|          hydro|  2486777.5|2021-01-01 02:00:00|
|2021-01-01 04:00:00|2024-12-20 10:35:40|      NO1|          hydro|  2461176.0|2021-01-01 03:00:00|
|2021-01-01 05:00:00|2024-12-20 10:35:40|      NO1|          hydro|  2466969.2|2021-01-01 04:00:00|
+-------------------+-------------------+---------+---------------+-----------+-------------------+
only showing top 5 rows



25/11/26 22:39:29 WARN TaskSetManager: Stage 3 contains a task of very large size (11700 KiB). The maximum recommended task size is 1000 KiB.


Production data successfully inserted into Cassandra!
Spark DataFrame schema before write:
root
 |-- consumptiongroup: string (nullable = true)
 |-- endtime: timestamp (nullable = true)
 |-- lastupdatedtime: timestamp (nullable = true)
 |-- meteringpointcount: long (nullable = true)
 |-- pricearea: string (nullable = true)
 |-- quantitykwh: double (nullable = true)
 |-- starttime: timestamp (nullable = true)

First 5 rows:


25/11/26 22:39:39 WARN TaskSetManager: Stage 4 contains a task of very large size (12461 KiB). The maximum recommended task size is 1000 KiB.
25/11/26 22:39:43 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 4 (TID 11): Attempting to kill Python Worker


+----------------+-------------------+-------------------+------------------+---------+-----------+-------------------+
|consumptiongroup|            endtime|    lastupdatedtime|meteringpointcount|pricearea|quantitykwh|          starttime|
+----------------+-------------------+-------------------+------------------+---------+-----------+-------------------+
|           cabin|2021-01-01 01:00:00|2024-12-20 10:35:40|            100607|      NO1|  177071.56|2021-01-01 00:00:00|
|           cabin|2021-01-01 02:00:00|2024-12-20 10:35:40|            100607|      NO1|  171335.12|2021-01-01 01:00:00|
|           cabin|2021-01-01 03:00:00|2024-12-20 10:35:40|            100607|      NO1|  164912.02|2021-01-01 02:00:00|
|           cabin|2021-01-01 04:00:00|2024-12-20 10:35:40|            100607|      NO1|  160265.77|2021-01-01 03:00:00|
|           cabin|2021-01-01 05:00:00|2024-12-20 10:35:40|            100607|      NO1|  159828.69|2021-01-01 04:00:00|
+----------------+-------------------+--

Traceback (most recent call last):
  File "/var/folders/cb/h4grq88s6sjgm0cnxyzjm7xw0000gn/T/ipykernel_1709/2625658215.py", line 34, in <module>
    .save()
     ^^^^^^
  File "/Users/henrikengdal/Documents/GitHub/IND320-henrikengdal-project/.conda/lib/python3.11/site-packages/pyspark/sql/readwriter.py", line 1461, in save
    self._jwrite.save()
  File "/Users/henrikengdal/Documents/GitHub/IND320-henrikengdal-project/.conda/lib/python3.11/site-packages/py4j/java_gateway.py", line 1322, in __call__
    return_value = get_return_value(
                   ^^^^^^^^^^^^^^^^^
  File "/Users/henrikengdal/Documents/GitHub/IND320-henrikengdal-project/.conda/lib/python3.11/site-packages/pyspark/errors/exceptions/captured.py", line 185, in deco
    raise converted from None
pyspark.errors.exceptions.captured.IllegalArgumentException: Attempting to write to C* Table but missing
primary key columns: [productiongroup]


In [16]:
# Read the necessary columns from Cassandra
df_spark_read_production = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(table="production", keyspace=keyspace_name)
    .load()
    .select("pricearea", "productiongroup", "starttime", "quantitykwh")
)


df_pd = df_spark_read_production.toPandas()
print(f"Retrieved {len(df_pd)} rows from Cassandra.")

Retrieved 872953 rows from Cassandra.


In [ ]:
# Insert the Spark DataFrame into Cassandra with error handling and schema debug
try:
    print("Spark DataFrame schema before write:")
    df_spark_consumption.printSchema()
    print("First 5 rows:")
    df_spark_consumption.show(5)

    (
        df_spark_consumption.write
        .format("org.apache.spark.sql.cassandra")
        .mode("append")
        .options(table="consumption", keyspace=keyspace_name)
        .save()
    )
    print("Consumption data successfully inserted into Cassandra!")
except Exception as e:
    import traceback
    print("Error while writing to Cassandra:")
    traceback.print_exc()
    print(f"Exception: {e}")

# Read the necessary columns from Cassandra
df_spark_read_consumption = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(table="consumption", keyspace=keyspace_name)
    .load()
    .select("pricearea", "consumptiongroup", "starttime", "quantitykwh")
)

df_cs = df_spark_read_consumption.toPandas()
print(f"Retrieved {len(df_cs)} rows from Cassandra.")



AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `consumptiongroup` cannot be resolved. Did you mean one of the following? [`productiongroup`, `endtime`, `pricearea`, `quantitykwh`, `starttime`].;
'Project [pricearea#264, 'consumptiongroup, starttime#265, quantitykwh#269]
+- RelationV2[pricearea#264, starttime#265, productiongroup#266, endtime#267, lastupdatedtime#268, quantitykwh#269]  consumption


In [20]:


# Connect to MongoDB, reuse 'uri' from previous code
client = MongoClient(uri, server_api=ServerApi('1'))
db = client["Elhub_Data"]
collection = db["production"]

# Convert Spark DataFrame to Pandas
df_mongo = df_spark_read_production.toPandas()
# look at df_mongo.head()
print(df_mongo.head())

# Convert to list of dictionaries (MongoDB format)
records = df_mongo.to_dict("records")

# Insert only if collection is empty (prevents duplicates)
if collection.count_documents({}) == 0:
    collection.insert_many(records)
    print(f"Inserted {len(records)} new records into 'production'.")
else:
    print(f"Collection already contains data — skipping insert.")



  pricearea productiongroup  starttime  quantitykwh
0       NO3           hydro 2021-01-01  2836774.000
1       NO3           other 2021-01-01        0.000
2       NO3           solar 2021-01-01       19.722
3       NO3         thermal 2021-01-01        0.000
4       NO3            wind 2021-01-01   259312.200
Inserted 872953 new records into 'production'.
